In [9]:
"""
NB03_Perturbing_Acceleration_Sampling

Purpose
-------
1) Load nominal orbit samples from NB02 (.npz).
2) Evaluate g_SH in Moon-fixed frame using NB01 corrected evaluator.
3) Form disturbing acceleration:  dg = g_SH - g0  (Moon-fixed).
4) Rotate dg back to inertial and project into RTN (A_R, A_T, A_N).
5) Save all arrays to .npz artifact for NB04/NB05.

Dependencies
------------
Expects the following from NB00, NB01, NB02 (via shared kernel or %run):
  NB00: CONFIG
  NB01: sh (SHModel), gravity_accel_sph_harm()
  NB02: inertial_to_fixed(), fixed_to_inertial(), norm_rows(), unit_rows()

If these are not in the namespace, the notebook loads the corrected orbit
npz and provides clear error messages for missing functions.

Outputs
-------
artifacts/dg_samples_<tag>_L<degree>.npz  containing:
  nu, t, r_eci, v_eci, r_fixed,
  g_SH_fixed, g0_fixed, dg_fixed, dg_eci,
  AR, AT, AN, dg_mag, meta_json
"""

from __future__ import annotations

import json
import time
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

In [10]:
# ============================================================
# Cell 1 : Configuration and File Paths
# ============================================================

# --- Truncation degree ---
# Pull from CONFIG if available; otherwise use a sensible default.
try:
    L = CONFIG["L_default"]
except NameError:
    L = 200   # NB00-consistent default

# --- Locate orbit npz from corrected NB02 ---
# Try CONFIG-derived name first, then common fallback names.
try:
    _alt = CONFIG["default_orbit"]["a_km"] - CONFIG["R_moon_km"]
    _M   = CONFIG["M_default"]
except NameError:
    _alt = 50.0
    _M   = 4000

ORBIT_NPZ = Path(f"nominal_orbit_llo{int(_alt)}km_M{_M}.npz")
if not ORBIT_NPZ.exists():
    # Try old naming convention as last resort
    _fallback = Path(f"nominal_orbit_llo40km_M{_M}.npz")
    if _fallback.exists():
        print(f"WARNING: Using old orbit file {_fallback}. Re-run corrected NB02 for best results.")
        ORBIT_NPZ = _fallback
    else:
        raise FileNotFoundError(
            f"Missing {ORBIT_NPZ}. Run corrected NB02 first."
        )

ART_DIR = Path("./artifacts")
ART_DIR.mkdir(exist_ok=True)

RUN_TAG = datetime.now(timezone.utc).strftime("NB03_%Y%m%dT%H%M%SZ")
OUT_NPZ = ART_DIR / f"dg_samples_{RUN_TAG}_L{L}.npz"

print(f"Orbit input : {ORBIT_NPZ}")
print(f"Output      : {OUT_NPZ}")
print(f"Degree L    : {L}")

Orbit input : nominal_orbit_llo50km_M4000.npz
Output      : artifacts\dg_samples_NB03_20260316T033826Z_L200.npz
Degree L    : 200


In [11]:
# ============================================================
# Cell 2 : Load Orbit Data from Corrected NB02
# ============================================================
orb = np.load(ORBIT_NPZ)

nu       = orb["nu"]          # (M,)
t_grid   = orb["t"]           # (M,) seconds
r_eci    = orb["r_eci"]       # (M, 3) km
v_eci    = orb["v_eci"]       # (M, 3) km/s
r_fixed  = orb["r_fixed"]     # (M, 3) km — from corrected NB02

# Orbital parameters stored in the npz
mu_orb      = float(orb["mu"])
Rm_orb      = float(orb["Rm"])
omega_orb   = float(orb["omega_moon"])
a_orb       = float(orb["a"])
e_orb       = float(orb["e"])
i_rad_orb   = float(orb["i_rad"]) if "i_rad" in orb else float(orb.get("i", 0.0))
period_orb  = float(orb["period_s"])
p_orb       = float(orb["p"]) if "p" in orb else a_orb * (1.0 - e_orb**2)
h_orb       = float(orb["h"]) if "h" in orb else np.sqrt(mu_orb * p_orb)

# Load RTN basis if saved by corrected NB02
if "rhat" in orb and "that" in orb and "nhat" in orb:
    rhat_nb02 = orb["rhat"]
    that_nb02 = orb["that"]
    nhat_nb02 = orb["nhat"]
    _rtn_source = "NB02 npz"
else:
    rhat_nb02 = that_nb02 = nhat_nb02 = None
    _rtn_source = "will recompute"

M = nu.size

print("Loaded orbit:")
print(f"  File       = {ORBIT_NPZ}")
print(f"  M          = {M}")
print(f"  a [km]     = {a_orb}")
print(f"  e          = {e_orb}")
print(f"  i [deg]    = {np.degrees(i_rad_orb):.1f}")
print(f"  p [km]     = {p_orb}")
print(f"  h [km^2/s] = {h_orb:.6f}")
print(f"  T [s]      = {period_orb:.4f}  ({period_orb/3600:.4f} hrs)")
print(f"  mu [km^3/s^2] = {mu_orb}")
print(f"  omega [rad/s] = {omega_orb:.7e}")
print(f"  RTN source    = {_rtn_source}")

Loaded orbit:
  File       = nominal_orbit_llo50km_M4000.npz
  M          = 4000
  a [km]     = 1787.4
  e          = 0.0
  i [deg]    = 90.0
  p [km]     = 1787.4
  h [km^2/s] = 2960.281212
  T [s]      = 6780.9479  (1.8836 hrs)
  mu [km^3/s^2] = 4902.800076
  omega [rad/s] = 2.6616995e-06
  RTN source    = NB02 npz


In [13]:
# ============================================================
# Cell 3 : Ensure NB01 Gravity Evaluator is Available
# ============================================================
# Try to use gravity_accel_sph_harm from a prior %run of NB01.
# If not found, load the model and evaluator directly here.

_need_load = False
try:
    _ = gravity_accel_sph_harm
    _ = sh
    print("NB01 gravity evaluator found in namespace.")
except NameError:
    print("NB01 not in namespace — loading gravity model and evaluator directly.")
    _need_load = True

if _need_load:
    # --- Execute NB01's core code inline ---
    import math
    from dataclasses import dataclass

    try:
        from numba import njit
    except ImportError:
        def njit(*args, **kwargs):
            def wrapper(fn): return fn
            if len(args) == 1 and callable(args[0]): return args[0]
            return wrapper

    MODEL_DIR = Path("./gravity_models")
    LOCAL_SHA_PATH = MODEL_DIR / "gggrx_0900c_sha.tab"
    if not LOCAL_SHA_PATH.exists():
        raise FileNotFoundError(
            f"Gravity coefficient file not found: {LOCAL_SHA_PATH}\n"
            "Run NB01 at least once to download it."
        )

    # --- Minimal SHModel + parser (same as corrected NB01) ---
    @dataclass
    class SHModel:
        C: np.ndarray; S: np.ndarray
        sigmaC: np.ndarray; sigmaS: np.ndarray
        Lmax: int; mu_km3_s2: float; Rref_km: float
        normalized: bool; source: str

    def parse_gggrx_pds3_fixed(path):
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
        parts = [p.strip() for p in lines[0].split(",") if p.strip()]
        Rref_km = float(parts[0]); mu_km3_s2 = float(parts[1])
        Lmax = int(float(parts[3])); normalized = (int(float(parts[5])) == 1)
        C = np.zeros((Lmax+1, Lmax+1)); S = np.zeros_like(C)
        sigmaC = np.zeros_like(C); sigmaS = np.zeros_like(C)
        for ln in lines[2:]:
            if not ln.strip() or len(ln) < 107: continue
            try:
                l=int(ln[0:5]); m=int(ln[6:11])
                C[l,m]=float(ln[12:35]); S[l,m]=float(ln[36:59])
                sigmaC[l,m]=float(ln[60:83]); sigmaS[l,m]=float(ln[84:107])
            except: continue
        if C[0,0]==0.0: C[0,0]=1.0
        return SHModel(C=C,S=S,sigmaC=sigmaC,sigmaS=sigmaS,Lmax=Lmax,
                       mu_km3_s2=mu_km3_s2,Rref_km=Rref_km,
                       normalized=normalized,source=str(path))

    sh = parse_gggrx_pds3_fixed(LOCAL_SHA_PATH)

    # --- Corrected ALF recursion (geodetic, no Condon-Shortley) ---
    @njit(cache=True)
    def _fully_normalized_legendre(sin_phi, cos_phi, L):
        Pbar = np.zeros((L+1, L+1)); dPbar = np.zeros((L+1, L+1))
        Pbar[0,0] = 1.0
        for m in range(1, L+1):
            fac = math.sqrt((2.0*m+1.0)/(2.0*m))
            Pbar[m,m] = cos_phi * fac * Pbar[m-1,m-1]
            dPbar[m,m] = fac*(-sin_phi*Pbar[m-1,m-1] + cos_phi*dPbar[m-1,m-1])
        for m in range(0, L):
            fac = math.sqrt(2.0*m+3.0)
            Pbar[m+1,m] = sin_phi * fac * Pbar[m,m]
            dPbar[m+1,m] = fac*(cos_phi*Pbar[m,m] + sin_phi*dPbar[m,m])
        for m in range(0, L+1):
            for l in range(m+2, L+1):
                ls=float(l); ms=float(m)
                alpha = math.sqrt((4.0*ls*ls-1.0)/(ls*ls-ms*ms))
                beta = math.sqrt(((ls-1.0)**2-ms*ms)/(4.0*(ls-1.0)**2-1.0))
                Pbar[l,m] = alpha*(sin_phi*Pbar[l-1,m] - beta*Pbar[l-2,m])
                dPbar[l,m] = alpha*(cos_phi*Pbar[l-1,m] + sin_phi*dPbar[l-1,m] - beta*dPbar[l-2,m])
        return Pbar, dPbar

    EPS_POLE = 1e-10

    @njit(cache=True)
    def _sh_summation(C, S, Pbar, dPbar_dphi, L, mu, a_ref, r, lam, sin_phi, cos_phi):
        cosml = np.empty(L+1); sinml = np.empty(L+1)
        cosml[0]=1.0; sinml[0]=0.0
        c1=math.cos(lam); s1=math.sin(lam)
        for m in range(1, L+1):
            cosml[m]=cosml[m-1]*c1 - sinml[m-1]*s1
            sinml[m]=sinml[m-1]*c1 + cosml[m-1]*s1
        rho = a_ref/r; ar_acc=0.0; aphi_acc=0.0; alam_acc=0.0; rho_l=1.0
        for l in range(0, L+1):
            if l > 0: rho_l *= rho
            sV=0.0; sdphi=0.0; sdlam=0.0
            for m in range(0, l+1):
                trig = C[l,m]*cosml[m] + S[l,m]*sinml[m]
                sV += Pbar[l,m]*trig; sdphi += dPbar_dphi[l,m]*trig
                if m > 0: sdlam += Pbar[l,m]*m*(-C[l,m]*sinml[m]+S[l,m]*cosml[m])
            ar_acc += (l+1)*rho_l*sV; aphi_acc += rho_l*sdphi; alam_acc += rho_l*sdlam
        r2=r*r
        return -(mu/r2)*ar_acc, -(mu/r2)*aphi_acc, -(mu/r2)*(alam_acc/cos_phi)

    def gravity_accel_sph_harm(sh, r_bf_km, degree=50):
        x,y,z = float(r_bf_km[0]),float(r_bf_km[1]),float(r_bf_km[2])
        r = math.sqrt(x*x+y*y+z*z)
        lam = math.atan2(y,x); sin_phi=z/r
        cos_phi = math.sqrt(max(0.0, 1.0-sin_phi*sin_phi))
        if cos_phi < EPS_POLE:
            sin_phi = math.copysign(math.sqrt(1.0-EPS_POLE**2), sin_phi)
            cos_phi = EPS_POLE
        L = min(int(degree), sh.Lmax)
        Pbar, dP = _fully_normalized_legendre(sin_phi, cos_phi, L)
        ar,aphi,alam = _sh_summation(sh.C,sh.S,Pbar,dP,L,sh.mu_km3_s2,sh.Rref_km,r,lam,sin_phi,cos_phi)
        cl=math.cos(lam); sl=math.sin(lam)
        ax = ar*cos_phi*cl + aphi*(-sin_phi)*cl + alam*(-sl)
        ay = ar*cos_phi*sl + aphi*(-sin_phi)*sl + alam*cl
        az = ar*sin_phi + aphi*cos_phi
        return np.array([ax,ay,az], dtype=np.float64)

    print(f"  Loaded model: {sh.source}, Lmax={sh.Lmax}")

print(f"  Model GM     : {sh.mu_km3_s2}")
print(f"  Model Rref   : {sh.Rref_km}")
print(f"  Using degree : {L}  (clamped to {min(L, sh.Lmax)})")

NB01 not in namespace — loading gravity model and evaluator directly.
  Loaded model: gravity_models\gggrx_0900c_sha.tab, Lmax=900
  Model GM     : 4902.79996708864
  Model Rref   : 1738.0
  Using degree : 200  (clamped to 200)


In [15]:
# ============================================================
# Cell 4 : Ensure Frame Transforms are Available
# ============================================================
_need_frames = False
try:
    _ = inertial_to_fixed
    _ = fixed_to_inertial
    print("NB02 frame transforms found in namespace.")
except NameError:
    print("NB02 not in namespace — defining frame transforms directly.")
    _need_frames = True

if _need_frames:
    def _R3(theta):
        c=np.cos(theta); s=np.sin(theta)
        return np.array([[c,-s,0],[s,c,0],[0,0,1.0]])

    def inertial_to_fixed(r_eci, t, omega):
        r_fixed = np.empty_like(r_eci)
        for k in range(r_eci.shape[0]):
            r_fixed[k] = _R3(-omega * t[k]) @ r_eci[k]
        return r_fixed

    def fixed_to_inertial(vec_fixed, t, omega):
        vec_eci = np.empty_like(vec_fixed)
        for k in range(vec_fixed.shape[0]):
            vec_eci[k] = _R3(+omega * t[k]) @ vec_fixed[k]
        return vec_eci

# Cross-check against saved r_fixed
r_fixed_check = inertial_to_fixed(r_eci, t_grid, omega=omega_orb)
diff_fixed = np.max(np.abs(r_fixed_check - r_fixed))
print(f"Frame cross-check: max|recomputed - saved| = {diff_fixed:.2e} km")

if diff_fixed > 1e-10:
    print("WARNING: mismatch — using recomputed r_fixed.")
    r_fixed = r_fixed_check
else:
    print("Frame conventions consistent.")

NB02 not in namespace — defining frame transforms directly.
Frame cross-check: max|recomputed - saved| = 0.00e+00 km
Frame conventions consistent.


In [16]:
# ============================================================
# Cell 5 : Evaluate Disturbing Acceleration in Moon-Fixed Frame
# ============================================================
# dg = g_SH(r; L) - g0(r)
# where g0 = -mu * r / |r|^3  is the central two-body acceleration.
#
# Both are evaluated in the Moon-fixed frame (body-fixed),
# using the gravity model's native GM for consistency (Section 3.2).

def central_accel(mu_km3_s2: float, r_km: np.ndarray) -> np.ndarray:
    """Central (two-body) gravitational acceleration [km/s^2]."""
    rr = np.linalg.norm(r_km)
    return -mu_km3_s2 * r_km / (rr**3)


mu_grav = float(sh.mu_km3_s2)   # model-native GM for evaluator consistency

g_SH_fixed = np.empty((M, 3), dtype=np.float64)
g0_fixed   = np.empty((M, 3), dtype=np.float64)
dg_fixed   = np.empty((M, 3), dtype=np.float64)

t_start = time.time()
for k in range(M):
    g_SH_fixed[k] = gravity_accel_sph_harm(sh, r_fixed[k], degree=L)
    g0_fixed[k]   = central_accel(mu_grav, r_fixed[k])
    dg_fixed[k]   = g_SH_fixed[k] - g0_fixed[k]

    if (k + 1) % 500 == 0 or k == M - 1:
        elapsed = time.time() - t_start
        rate = (k + 1) / elapsed
        eta = (M - k - 1) / rate if rate > 0 else 0
        print(f"  [{k+1:5d}/{M}]  {elapsed:.1f}s elapsed,  {rate:.0f} pts/s,  ETA {eta:.0f}s",
              end="\r")

elapsed_total = time.time() - t_start
print(f"\nDone in {elapsed_total:.1f}s  ({M/elapsed_total:.0f} pts/s)")

dg_mag = np.linalg.norm(dg_fixed, axis=1)

print(f"\ndg magnitude [km/s^2]:")
print(f"  min  = {dg_mag.min():.6e}")
print(f"  mean = {dg_mag.mean():.6e}")
print(f"  max  = {dg_mag.max():.6e}")
print(f"  max / (mu/r^2) = {dg_mag.max() / (mu_grav / a_orb**2):.6e}  (relative to central)")

  [ 4000/4000]  2.0s elapsed,  1962 pts/s,  ETA 0s
Done in 2.0s  (1962 pts/s)

dg magnitude [km/s^2]:
  min  = 1.941380e-07
  mean = 7.142456e-07
  max  = 1.701346e-06
  max / (mu/r^2) = 1.108644e-03  (relative to central)


In [17]:
# ============================================================
# Cell 6 : Rotate dg to Inertial Frame
# ============================================================
# The gravity evaluator returns g_SH and g0 in Moon-fixed.
# dg is therefore also Moon-fixed.
# To project into RTN (defined in inertial), rotate dg to inertial:
#   dg_eci = R3(+omega*t) @ dg_fixed   (fixed_to_inertial)

dg_eci = fixed_to_inertial(dg_fixed, t_grid, omega=omega_orb)

dg_eci_mag = np.linalg.norm(dg_eci, axis=1)

# Magnitude must be preserved by rotation
mag_err = np.max(np.abs(dg_mag - dg_eci_mag))
print(f"Rotated dg to inertial.")
print(f"  max | |dg_fixed| - |dg_eci| | = {mag_err:.2e}  (expect ~0)")

Rotated dg to inertial.
  max | |dg_fixed| - |dg_eci| | = 2.12e-22  (expect ~0)


In [18]:
# ============================================================
# Cell 7 : Project dg into RTN Components
# ============================================================
# Use RTN basis from NB02 npz if available; otherwise recompute.

if rhat_nb02 is not None:
    rhat = rhat_nb02
    that = that_nb02
    nhat = nhat_nb02
    print("Using RTN basis from NB02 npz.")
else:
    # Recompute (should match NB02)
    def _norm_rows(X):
        return np.linalg.norm(X, axis=1)
    def _unit_rows(X):
        return X / _norm_rows(X)[:, None]

    rhat = _unit_rows(r_eci)
    h_vec = np.cross(r_eci, v_eci)
    nhat = _unit_rows(h_vec)
    that = _unit_rows(np.cross(nhat, rhat))
    print("Recomputed RTN basis from r_eci, v_eci.")

# Project
AR = np.sum(dg_eci * rhat, axis=1)
AT = np.sum(dg_eci * that, axis=1)
AN = np.sum(dg_eci * nhat, axis=1)

# Verification: |dg_eci|^2 should equal AR^2 + AT^2 + AN^2
dg_rtn_mag = np.sqrt(AR**2 + AT**2 + AN**2)
recon_err  = np.max(np.abs(dg_eci_mag - dg_rtn_mag))

print(f"\nRTN projection:")
print(f"  AR  min / max = {AR.min():.6e} / {AR.max():.6e}")
print(f"  AT  min / max = {AT.min():.6e} / {AT.max():.6e}")
print(f"  AN  min / max = {AN.min():.6e} / {AN.max():.6e}")
print(f"  Reconstruction error max||dg| - sqrt(AR^2+AT^2+AN^2)| = {recon_err:.2e}")

Using RTN basis from NB02 npz.

RTN projection:
  AR  min / max = -1.171520e-06 / 1.661561e-06
  AT  min / max = -9.555580e-07 / 8.026571e-07
  AN  min / max = -4.811331e-07 / 7.264885e-07
  Reconstruction error max||dg| - sqrt(AR^2+AT^2+AN^2)| = 4.24e-22


In [19]:
# ============================================================
# Cell 8 : Save Artifact
# ============================================================
meta = {
    "run_tag": RUN_TAG,
    "L": int(L),
    "orbit_npz": str(ORBIT_NPZ),
    "orbit_params": {
        "a_km": a_orb,
        "e": e_orb,
        "i_rad": i_rad_orb,
        "p_km": p_orb,
        "h_km2_s": h_orb,
        "period_s": period_orb,
        "mu_orbit": mu_orb,
        "Rm_km": Rm_orb,
        "omega_rad_s": omega_orb,
    },
    "gravity_model": {
        "source": sh.source,
        "Lmax": int(sh.Lmax),
        "L_used": int(L),
        "normalized": bool(sh.normalized),
        "GM_km3_s2": float(sh.mu_km3_s2),
        "Rref_km": float(sh.Rref_km),
    },
    "notes": [
        "dg_fixed = g_SH(r_fixed; L) - g0(r_fixed) using model-native GM.",
        "dg_eci = fixed_to_inertial(dg_fixed) using corrected NB02 convention R3(+omega*t).",
        "RTN projection: AR = dg_eci . rhat,  AT = dg_eci . that,  AN = dg_eci . nhat.",
        "RTN basis from NB02 npz (or recomputed if not available).",
    ],
}

np.savez(
    OUT_NPZ,
    # Grid
    nu=nu,
    t=t_grid,
    # States
    r_eci=r_eci,
    v_eci=v_eci,
    r_fixed=r_fixed,
    # Gravity
    g_SH_fixed=g_SH_fixed,
    g0_fixed=g0_fixed,
    dg_fixed=dg_fixed,
    dg_eci=dg_eci,
    # RTN components
    AR=AR,
    AT=AT,
    AN=AN,
    # Scalar magnitudes
    dg_mag=dg_mag,
    # RTN basis (for downstream use)
    rhat=rhat,
    that=that,
    nhat=nhat,
    # Metadata
    meta_json=json.dumps(meta),
)

print(f"Saved: {OUT_NPZ}")
print(f"Keys : {sorted(np.load(OUT_NPZ).files)}")
print(f"\n--- NB03 complete ---")

Saved: artifacts\dg_samples_NB03_20260316T033826Z_L200.npz
Keys : ['AN', 'AR', 'AT', 'dg_eci', 'dg_fixed', 'dg_mag', 'g0_fixed', 'g_SH_fixed', 'meta_json', 'nhat', 'nu', 'r_eci', 'r_fixed', 'rhat', 't', 'that', 'v_eci']

--- NB03 complete ---
